## Current research direction — HRI adaptation

Reusable preparation: review and adapt these steps for endemic-bird HRI assessment. Any legacy forest-loss target analysis in this notebook is exploratory, not an HRI definition.

The original code and saved outputs are retained. No cells were executed during restoration. Use `docs/methodology/hri_workflow.md` for the current direction and `notebooks/hri/` for the simple lecturer-facing versions. Do not describe existing forest-loss outputs as habitat-risk classes.

**HRI execution route:** the original cells below are preserved historical preparation. For the corrected outputs, start at **HRI preparation — visible cleaning and species join** near the end and run that section in order. It is self-contained. Earlier consecutive-year forest-loss feasibility checks are not part of the HRI target. Do not run both routes indiscriminately.

Run from the project root or `notebooks/` using the project `.venv` kernel. Raw input is in `data/raw/observations/`; review tables are in `data/interim/`; source notes are in `data/metadata/`. Existing outputs are from the pre-reorganization run; rerun to refresh displayed paths.

In [1]:
from pathlib import Path
import pandas as pd

# Resolve the project from either the root or notebooks/ working directory.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "requirements-research-lock.txt").is_file()
     and (folder / "data/raw/observations/final_dataset_complete.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the component4 project folder.")

RAW_DATA = PROJECT_ROOT / "data/raw/observations/final_dataset_complete.csv"

df_raw = pd.read_csv(RAW_DATA)
df_clean = df_raw.copy(deep=True)

# Preserve the original row position for tracing issues.
df_clean.insert(0, "source_row_id", df_raw.index)

print("Original rows:", len(df_raw))
print("Working-copy rows:", len(df_clean))

Original rows: 102929
Working-copy rows: 102929


In [2]:
# Preserve the original date text.
df_clean["observation_date_raw"] = df_clean["observation_date"]

df_clean["observation_date"] = pd.to_datetime(
    df_clean["observation_date_raw"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Keep observation_count unchanged and add a numeric version.
df_clean["observation_count_numeric"] = pd.to_numeric(
    df_clean["observation_count"],
    errors="coerce"
)

print("Unreadable dates:", df_clean["observation_date"].isna().sum())
print(
    "Missing numeric counts:",
    df_clean["observation_count_numeric"].isna().sum()
)

df_clean[
    [
        "source_row_id",
        "observation_date",
        "observation_count",
        "observation_count_numeric"
    ]
].head()

Unreadable dates: 0
Missing numeric counts: 12743


,source_row_id,observation_date,observation_count,observation_count_numeric
0,0,2010-01-07,1,1.0
1,1,2010-01-07,1,1.0
2,2,2010-01-07,1,1.0
3,3,2010-01-07,1,1.0
4,4,2010-01-07,1,1.0


In [3]:
print("Before:", df_raw.shape)
print("After: ", df_clean.shape)

print(
    df_clean[
        ["observation_date", "observation_count_numeric"]
    ].dtypes
)

Before: (102929, 23)
After:  (102929, 26)
observation_date             datetime64[us]
observation_count_numeric           float64
dtype: object


In [4]:
# Compare original columns only: source_row_id is unique for every row.
df_clean["flag_exact_duplicate"] = df_raw.duplicated(keep=False)

# Flag forest loss recorded in a later year than the observation.
df_clean["flag_loss_after_observation"] = (
    df_clean["forest_loss_year"] > 0
) & (
    df_clean["forest_loss_year"]
    > df_clean["observation_date"].dt.year
)

# Flag loss in the same year: its timing within that year is unknown.
df_clean["flag_loss_same_year"] = (
    df_clean["forest_loss_year"] > 0
) & (
    df_clean["forest_loss_year"]
    == df_clean["observation_date"].dt.year
)

# Flag observation durations exceeding 24 hours for source review.
df_clean["flag_duration_over_24h"] = (
    df_clean["duration_minutes"] > 1440
)

flag_columns = [
    "flag_exact_duplicate",
    "flag_loss_after_observation",
    "flag_loss_same_year",
    "flag_duration_over_24h"
]

df_clean[flag_columns].sum().to_frame("flagged_rows")

,flagged_rows
flag_exact_duplicate,31667
flag_loss_after_observation,47
flag_loss_same_year,37
flag_duration_over_24h,20


In [5]:
duration_review = df_clean.loc[
    df_clean["flag_duration_over_24h"],
    [
        "source_row_id",
        "common_name",
        "observation_date",
        "protocol",
        "duration_minutes",
        "effort_distance_km",
        "flag_exact_duplicate"
    ]
].sort_values("duration_minutes", ascending=False)

print("Rows needing duration review:", len(duration_review))

duration_review

Rows needing duration review: 20


,source_row_id,common_name,observation_date,protocol,duration_minutes,effort_distance_km,flag_exact_duplicate
43971,43971,Sri Lanka Woodshrike,2020-03-07,Traveling,2880.0,30.000,False
43970,43970,Sri Lanka Junglefowl,2020-03-07,Traveling,2880.0,30.000,False
5729,5729,Red-backed Flameback,2013-11-12,Traveling,1500.0,12.875,False
5836,5836,Yellow-fronted Barbet,2013-11-12,Traveling,1500.0,12.875,False
5824,5824,White-throated Flowerpecker,2013-11-12,Traveling,1500.0,12.875,False
5817,5817,Sri Lanka Drongo,2013-11-12,Traveling,1500.0,12.875,False
5793,5793,Red-faced Malkoha,2013-11-12,Traveling,1500.0,12.875,False
5787,5787,Sri Lanka Green-Pigeon,2013-11-12,Traveling,1500.0,12.875,False
5785,5785,Layard's Parakeet,2013-11-12,Traveling,1500.0,12.875,False
5778,5778,Green-billed Coucal,2013-11-12,Traveling,1500.0,12.875,False


In [6]:
environment_outlier_columns = [
    "temperature_mean",
    "precipitation_sum",
    "windspeed_max",
    "windgusts_max"
]

outlier_limits = []

for column in environment_outlier_columns:
    values = df_clean[column]

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    flag_name = f"flag_iqr_{column}"

    df_clean[flag_name] = (
        values.notna()
        & ((values < lower) | (values > upper))
    )

    outlier_limits.append({
        "variable": column,
        "lower_limit": lower,
        "upper_limit": upper,
        "flagged_rows": df_clean[flag_name].sum()
    })

outlier_limits = pd.DataFrame(outlier_limits)

outlier_limits.round(2)

,variable,lower_limit,upper_limit,flagged_rows
0,temperature_mean,19.80,31.00,4329
1,precipitation_sum,-9.30,16.30,8413
2,windspeed_max,-4.15,29.05,1079
3,windgusts_max,2.50,60.10,1200


In [7]:
from pathlib import Path

output_folder = PROJECT_ROOT / "data/interim"
output_folder.mkdir(parents=True, exist_ok=True)

# Confirm that no rows were accidentally removed.
assert len(df_clean) == len(df_raw)
assert df_clean["source_row_id"].is_unique

# Save the prepared dataset to a new file.
df_clean.to_csv(
    output_folder / "dataset_review_stage.csv",
    index=False
)

# Save the count for every review flag.
flag_columns = [
    column for column in df_clean.columns
    if column.startswith("flag_")
]

review_summary = (
    df_clean[flag_columns]
    .sum()
    .rename("flagged_rows")
    .to_frame()
)

review_summary.to_csv(
    output_folder / "review_flag_summary.csv"
)

# Save the statistical limits used for environmental outlier flags.
outlier_limits.to_csv(
    output_folder / "eda_outlier_limits.csv",
    index=False
)

print("Saved to:", output_folder.resolve())
print("Dataset shape:", df_clean.shape)
review_summary

Saved to: /Users/keerthikeerthi/Desktop/component4/data/interim
Dataset shape: (102929, 34)


,flagged_rows
flag_exact_duplicate,31667
flag_loss_after_observation,47
flag_loss_same_year,37
flag_duration_over_24h,20
flag_iqr_temperature_mean,4329
flag_iqr_precipitation_sum,8413
flag_iqr_windspeed_max,1079
flag_iqr_windgusts_max,1200


In [8]:
# Work in a separate table.
coverage_data = df_clean[
    ["latitude", "longitude", "observation_date"]
].copy()

coverage_data["observation_year"] = (
    coverage_data["observation_date"].dt.year
)

coverage_data["observation_day"] = (
    coverage_data["observation_date"].dt.normalize()
)

# One summary row per coordinate pair and year.
location_year_coverage = (
    coverage_data
    .groupby(["latitude", "longitude", "observation_year"])
    .agg(
        record_count=("observation_day", "size"),
        observed_days=("observation_day", "nunique")
    )
    .reset_index()
)

print("Location–year combinations:", len(location_year_coverage))

print("\nSummary of distinct observed days:")
print(location_year_coverage["observed_days"].describe())

location_year_coverage.head(10)

Location–year combinations: 9429

Summary of distinct observed days:
count    9429.000000
mean        2.786085
std         9.324408
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       275.000000
Name: observed_days, dtype: float64


,latitude,longitude,observation_year,record_count,observed_days
0,5.927466,80.588901,2021,1,1
1,5.927466,80.588901,2022,2,1
2,5.937312,80.573289,2016,3,1
3,5.937809,80.474420,2019,1,1
4,5.938094,80.576134,2020,2,1
5,5.938355,80.510221,2022,1,1
6,5.939328,80.529400,2019,1,1
7,5.939364,80.509408,2023,4,1
8,5.939497,80.570511,2019,1,1
9,5.939706,80.575097,2020,3,2


In [9]:
# Use the location–year table from the previous cell.
years_per_location = (
    location_year_coverage
    .groupby(["latitude", "longitude"])["observation_year"]
    .nunique()
    .rename("observed_years")
)

coverage_summary = (
    years_per_location
    .value_counts()
    .sort_index()
    .rename_axis("Number of observed years")
    .to_frame("Number of locations")
)

print(coverage_summary)

print(
    "\nLocations observed in at least two years:",
    (years_per_location >= 2).sum()
)

                          Number of locations
Number of observed years                     
1                                        6347
2                                         508
3                                         170
4                                          78
5                                          32
6                                          20
7                                          10
8                                          11
9                                          13
10                                          9
11                                         13
12                                          6
13                                          8
14                                         20

Locations observed in at least two years: 898


In [10]:
# Keep one record per location and observed year.
location_years = location_year_coverage[
    ["latitude", "longitude", "observation_year"]
].drop_duplicates()

# Prepare a second table representing the following year.
following_years = location_years.rename(
    columns={"observation_year": "next_year"}
)

# Match each location-year to the same location in the next year.
consecutive_pairs = (
    location_years
    .assign(next_year=lambda table: table["observation_year"] + 1)
    .merge(
        following_years,
        on=["latitude", "longitude", "next_year"],
        how="inner",
        validate="one_to_one"
    )
)

print("Consecutive location–year pairs:", len(consecutive_pairs))

print(
    "Locations with at least one consecutive pair:",
    len(consecutive_pairs[["latitude", "longitude"]].drop_duplicates())
)

consecutive_pairs.head(10)

Consecutive location–year pairs: 1661
Locations with at least one consecutive pair: 694


,latitude,longitude,observation_year,next_year
0,5.927466,80.588901,2021,2022
1,5.942950,80.560593,2020,2021
2,5.943487,80.515245,2021,2022
3,5.943487,80.515245,2022,2023
4,5.954918,80.537566,2018,2019
5,5.954918,80.537566,2019,2020
6,5.954918,80.537566,2020,2021
7,5.954918,80.537566,2021,2022
8,5.964167,80.522276,2021,2022
9,5.964167,80.522276,2022,2023


In [11]:
forest_columns = [
    "treecover2000",
    "forest_loss",
    "forest_loss_year"
]

# Check whether forest values are consistent at each coordinate pair.
forest_variation = (
    df_clean.groupby(["latitude", "longitude"])[forest_columns]
    .nunique(dropna=False)
)

conflicting_locations = forest_variation.gt(1).any(axis=1)

print("Locations with conflicting forest values:",
      conflicting_locations.sum())

if conflicting_locations.any():
    print("Review conflicting values before making a location-level table.")
else:
    forest_locations = df_clean[
        ["latitude", "longitude"] + forest_columns
    ].drop_duplicates()

    print("Distinct locations:", len(forest_locations))
    print(
        "Locations with recorded loss:",
        forest_locations["forest_loss"].eq(1).sum()
    )

    print("\nLocations by recorded loss year:")
    print(
        forest_locations.loc[
            forest_locations["forest_loss"].eq(1),
            "forest_loss_year"
        ].value_counts().sort_index()
    )

Locations with conflicting forest values: 0
Distinct locations: 7245
Locations with recorded loss: 180

Locations by recorded loss year:
forest_loss_year
2001     1
2002     2
2003     2
2004     2
2005     4
2006    31
2007     9
2008    22
2009     9
2010     7
2011     9
2012     4
2013     5
2014     5
2015     5
2016    15
2017     7
2018     8
2019     8
2020     7
2021     7
2022     3
2023     5
2024     3
Name: count, dtype: int64


In [12]:
# Attach the recorded loss year to each consecutive-year pair.
pair_review = consecutive_pairs.merge(
    forest_locations[
        ["latitude", "longitude", "forest_loss_year"]
    ],
    on=["latitude", "longitude"],
    how="left",
    validate="many_to_one"
)

# Identify pairs where loss had already been recorded by the first year.
pair_review["loss_by_start_year"] = (
    pair_review["forest_loss_year"].gt(0)
    & pair_review["forest_loss_year"].le(
        pair_review["observation_year"]
    )
)

# Identify loss recorded in the following year.
pair_review["recorded_loss_next_year"] = (
    pair_review["forest_loss_year"].eq(pair_review["next_year"])
)

# Inspect pairs without loss already recorded by the first year.
candidate_pairs = pair_review.loc[
    ~pair_review["loss_by_start_year"]
].copy()

print("Pairs without previously recorded loss:", len(candidate_pairs))
print(
    "Pairs with recorded loss in the following year:",
    candidate_pairs["recorded_loss_next_year"].sum()
)

candidate_pairs.loc[
    candidate_pairs["recorded_loss_next_year"],
    ["latitude", "longitude", "observation_year",
     "next_year", "forest_loss_year"]
]

Pairs without previously recorded loss: 1638
Pairs with recorded loss in the following year: 0


,latitude,longitude,observation_year,next_year,forest_loss_year


In [13]:
from pathlib import Path

source_notes = pd.DataFrame([
    {
        "data_group": "Bird observations",
        "reported_source": "eBird",
        "pending_verification": "Original export details and checklist IDs"
    },
    {
        "data_group": "Forest",
        "reported_source": "Hansen GFC 2024 v1.12",
        "pending_verification": (
            "QGIS extraction method, masking and loss-year conversion"
        )
    },
    {
        "data_group": "Weather",
        "reported_source": "Kaggle; exact dataset unknown",
        "pending_verification": (
            "Dataset link, original provider, units, "
            "time interval and spatial matching"
        )
    },
    {
        "data_group": "Elevation",
        "reported_source": "Kaggle; exact dataset unknown",
        "pending_verification": (
            "Dataset link, units, resolution and extraction method"
        )
    }
])

metadata_folder = PROJECT_ROOT / "data/metadata"
metadata_folder.mkdir(parents=True, exist_ok=True)

source_notes.to_csv(
    metadata_folder / "source_verification_notes.csv",
    index=False
)

source_notes

,data_group,reported_source,pending_verification
0,Bird observations,eBird,Original export details and checklist IDs
1,Forest,Hansen GFC 2024 v1.12,"QGIS extraction method, masking and loss-year ..."
2,Weather,Kaggle; exact dataset unknown,"Dataset link, original provider, units, time i..."
3,Elevation,Kaggle; exact dataset unknown,"Dataset link, units, resolution and extraction..."


In [14]:
# One row per exact coordinate pair.
location_reference = (
    df_clean[["latitude", "longitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)

location_reference.insert(
    0,
    "location_id",
    [f"LOC_{i:06d}" for i in range(1, len(location_reference) + 1)]
)

# Attach the location ID to every observation.
# Dropping an existing ID allows this cell to be rerun.
rows_before = len(df_clean)

df_clean = (
    df_clean.drop(columns=["location_id"], errors="ignore")
    .merge(
        location_reference,
        on=["latitude", "longitude"],
        how="left",
        validate="many_to_one",
        sort=False
    )
)

assert len(df_clean) == rows_before
assert df_clean["location_id"].notna().all()

location_reference.to_csv(
    output_folder / "location_reference.csv",
    index=False
)

print("Distinct locations:", len(location_reference))
print("Observation rows retained:", len(df_clean))

location_reference.head()

Distinct locations: 7245
Observation rows retained: 102929


,location_id,latitude,longitude
0,LOC_000001,5.927466,80.588901
1,LOC_000002,5.937312,80.573289
2,LOC_000003,5.937809,80.474420
3,LOC_000004,5.938094,80.576134
4,LOC_000005,5.938355,80.510221


In [15]:
forest_fields = [
    "treecover2000",
    "forest_loss",
    "forest_loss_year"
]

# Confirm that each location has consistent forest values.
variation = (
    df_clean.groupby("location_id")[forest_fields]
    .nunique(dropna=False)
)

assert not variation.gt(1).any().any(), (
    "Some locations have conflicting forest values. Review before continuing."
)

# Keep one set of forest measurements per location.
location_forest = (
    df_clean[["location_id"] + forest_fields]
    .drop_duplicates()
    .sort_values("location_id")
    .reset_index(drop=True)
)

assert location_forest["location_id"].is_unique
assert len(location_forest) == len(location_reference)

location_forest.to_csv(
    output_folder / "location_forest_review.csv",
    index=False
)

print("Locations in forest table:", len(location_forest))
print(
    "Locations with recorded loss:",
    location_forest["forest_loss"].eq(1).sum()
)

location_forest.head()

Locations in forest table: 7245
Locations with recorded loss: 180


,location_id,treecover2000,forest_loss,forest_loss_year
0,LOC_000001,0,0,0
1,LOC_000002,0,0,0
2,LOC_000003,27,0,0
3,LOC_000004,9,0,0
4,LOC_000005,0,0,0


In [16]:
species_location = (
    df_clean
    .groupby(
        ["location_id", "scientific_name", "common_name"],
        dropna=False
    )
    .agg(
        first_observed=("observation_date", "min"),
        last_observed=("observation_date", "max"),
        observed_days=("observation_date", "nunique"),
        record_count=("source_row_id", "size")
    )
    .reset_index()
)

species_location.to_csv(
    output_folder / "species_location_review.csv",
    index=False
)

print("Species–location combinations:", len(species_location))

species_location.head(10)

Species–location combinations: 24398


,location_id,scientific_name,common_name,first_observed,last_observed,observed_days,record_count
0,LOC_000001,Dinopium psarodes,Red-backed Flameback,2021-03-04,2022-01-22,2,2
1,LOC_000001,Psilopogon rubricapillus,Crimson-fronted Barbet,2022-01-22,2022-01-22,1,1
2,LOC_000002,Chrysocolaptes stricklandi,Crimson-backed Flameback,2016-03-11,2016-03-11,1,1
3,LOC_000002,Loriculus beryllinus,Sri Lanka Hanging-Parrot,2016-03-11,2016-03-11,1,1
4,LOC_000002,Tephrodornis affinis,Sri Lanka Woodshrike,2016-03-11,2016-03-11,1,1
5,LOC_000003,Dinopium psarodes,Red-backed Flameback,2019-02-24,2019-02-24,1,1
6,LOC_000004,Dinopium psarodes,Red-backed Flameback,2020-03-03,2020-03-03,1,1
7,LOC_000004,Ocyceros gingalensis,Sri Lanka Gray Hornbill,2020-03-03,2020-03-03,1,1
8,LOC_000005,Dinopium psarodes,Red-backed Flameback,2022-02-15,2022-02-15,1,1
9,LOC_000006,Dinopium psarodes,Red-backed Flameback,2019-02-01,2019-02-01,1,1


In [17]:
weather_columns = [
    "temperature_min",
    "temperature_mean",
    "temperature_max",
    "precipitation_sum",
    "rain_sum",
    "windspeed_max",
    "windgusts_max"
]

# Count distinct values within each location–date group.
weather_variation = (
    df_clean
    .groupby(["location_id", "observation_date"])[weather_columns]
    .nunique(dropna=False)
)

# A group is flagged if any weather column has multiple values.
conflict_mask = weather_variation.gt(1).any(axis=1)

print("Location–date groups checked:", len(weather_variation))
print("Groups with conflicting weather:", conflict_mask.sum())

weather_variation.loc[conflict_mask].head(10)

Location–date groups checked: 26270
Groups with conflicting weather: 0


,,temperature_min,temperature_mean,temperature_max,precipitation_sum,rain_sum,windspeed_max,windgusts_max
location_id,observation_date,,,,,,,


In [18]:
# Stop if the previous consistency check found conflicts.
assert not conflict_mask.any(), "Resolve weather conflicts first."

weather_reference = (
    df_clean[
        ["location_id", "observation_date"] + weather_columns
    ]
    .drop_duplicates()
    .sort_values(["location_id", "observation_date"])
    .reset_index(drop=True)
)

# Confirm one weather record per location and date.
assert not weather_reference.duplicated(
    ["location_id", "observation_date"]
).any()

weather_reference.to_csv(
    output_folder / "location_date_weather_review.csv",
    index=False
)

print("Weather table rows:", len(weather_reference))

weather_reference.head(10)

Weather table rows: 26270


,location_id,observation_date,temperature_min,temperature_mean,temperature_max,precipitation_sum,rain_sum,windspeed_max,windgusts_max
0,LOC_000001,2021-03-04,24.3,27.2,30.9,0.0,0.0,16.2,35.6
1,LOC_000001,2022-01-22,21.8,25.8,29.1,0.8,0.8,11.0,23.4
2,LOC_000002,2016-03-11,25.8,29.2,33.9,0.6,0.6,12.7,32.0
3,LOC_000003,2019-02-24,24.2,27.1,31.3,0.2,0.2,15.3,32.4
4,LOC_000004,2020-03-03,25.6,28.2,32.3,1.4,1.4,13.0,36.7
5,LOC_000005,2022-02-15,24.1,26.7,29.3,0.8,0.8,10.9,23.0
6,LOC_000006,2019-02-01,23.6,26.6,30.8,0.1,0.1,8.4,28.8
7,LOC_000007,2023-03-19,24.1,26.3,28.9,11.3,11.3,11.9,25.2
8,LOC_000008,2019-01-10,24.1,27.0,30.3,0.6,0.6,10.8,29.5
9,LOC_000009,2020-02-19,23.3,27.3,31.2,0.0,0.0,16.9,32.8


In [19]:
# Confirm that every original record is still present.
assert len(df_clean) == len(df_raw)
assert df_clean["source_row_id"].is_unique
assert df_clean["location_id"].notna().all()

# Confirm every observation links to a known location.
assert df_clean["location_id"].isin(
    location_reference["location_id"]
).all()

# Save the latest working table.
df_clean.to_csv(
    output_folder / "dataset_review_stage.csv",
    index=False
)

# Refresh the review-flag counts.
flag_columns = [
    column for column in df_clean.columns
    if column.startswith("flag_")
]

df_clean[flag_columns].sum().rename(
    "flagged_rows"
).to_csv(output_folder / "review_flag_summary.csv")

print("Saved observation rows:", len(df_clean))
print("Locations:", df_clean["location_id"].nunique())
print("Saved to:", output_folder.resolve())

Saved observation rows: 102929
Locations: 7245
Saved to: /Users/keerthikeerthi/Desktop/component4/data/interim


In [20]:
observation_year = df_clean["observation_date"].dt.year
loss_year = df_clean["forest_loss_year"]

df_clean["forest_loss_timing"] = "unknown"

df_clean.loc[
    loss_year.eq(0),
    "forest_loss_timing"
] = "no_mapped_loss"

df_clean.loc[
    loss_year.gt(0) & loss_year.lt(observation_year),
    "forest_loss_timing"
] = "before_observation_year"

df_clean.loc[
    loss_year.gt(0) & loss_year.eq(observation_year),
    "forest_loss_timing"
] = "same_year_order_unknown"

df_clean.loc[
    loss_year.gt(observation_year),
    "forest_loss_timing"
] = "after_observation_year"

# Calculate a year difference only for loss in an earlier year.
df_clean["years_since_prior_loss"] = (
    observation_year - loss_year
).where(
    df_clean["forest_loss_timing"].eq("before_observation_year")
).astype("Int64")

print(df_clean["forest_loss_timing"].value_counts())

df_clean[
    [
        "observation_date",
        "forest_loss_year",
        "forest_loss_timing",
        "years_since_prior_loss"
    ]
].head(10)

forest_loss_timing
no_mapped_loss             101357
before_observation_year      1488
after_observation_year         47
same_year_order_unknown        37
Name: count, dtype: int64


,observation_date,forest_loss_year,forest_loss_timing,years_since_prior_loss
0,2010-01-07,0,no_mapped_loss,<NA>
1,2010-01-07,0,no_mapped_loss,<NA>
2,2010-01-07,0,no_mapped_loss,<NA>
3,2010-01-07,0,no_mapped_loss,<NA>
4,2010-01-07,0,no_mapped_loss,<NA>
5,2010-01-07,0,no_mapped_loss,<NA>
6,2010-01-04,0,no_mapped_loss,<NA>
7,2010-01-11,0,no_mapped_loss,<NA>
8,2010-01-09,0,no_mapped_loss,<NA>
9,2010-01-09,0,no_mapped_loss,<NA>


In [21]:
# Start with one weather record per location and observation date.
environment_review = weather_reference.merge(
    location_forest,
    on="location_id",
    how="left",
    validate="many_to_one"
)

# Attach coordinates.
environment_review = environment_review.merge(
    location_reference,
    on="location_id",
    how="left",
    validate="many_to_one"
)

# Attach the descriptive forest-timing fields.
timing_fields = df_clean[
    [
        "location_id",
        "observation_date",
        "forest_loss_timing",
        "years_since_prior_loss"
    ]
].drop_duplicates()

assert not timing_fields.duplicated(
    ["location_id", "observation_date"]
).any()

environment_review = environment_review.merge(
    timing_fields,
    on=["location_id", "observation_date"],
    how="left",
    validate="one_to_one"
)

assert len(environment_review) == len(weather_reference)

environment_review.to_csv(
    output_folder / "environment_location_date_review.csv",
    index=False
)

# Also save the latest observation table with its new timing fields.
df_clean.to_csv(
    output_folder / "dataset_review_stage.csv",
    index=False
)

print("Environmental review rows:", len(environment_review))
environment_review.head()

Environmental review rows: 26270


,location_id,observation_date,temperature_min,temperature_mean,temperature_max,precipitation_sum,rain_sum,windspeed_max,windgusts_max,treecover2000,forest_loss,forest_loss_year,latitude,longitude,forest_loss_timing,years_since_prior_loss
0,LOC_000001,2021-03-04,24.3,27.2,30.9,0.0,0.0,16.2,35.6,0,0,0,5.927466,80.588901,no_mapped_loss,<NA>
1,LOC_000001,2022-01-22,21.8,25.8,29.1,0.8,0.8,11.0,23.4,0,0,0,5.927466,80.588901,no_mapped_loss,<NA>
2,LOC_000002,2016-03-11,25.8,29.2,33.9,0.6,0.6,12.7,32.0,0,0,0,5.937312,80.573289,no_mapped_loss,<NA>
3,LOC_000003,2019-02-24,24.2,27.1,31.3,0.2,0.2,15.3,32.4,27,0,0,5.937809,80.474420,no_mapped_loss,<NA>
4,LOC_000004,2020-03-03,25.6,28.2,32.3,1.4,1.4,13.0,36.7,9,0,0,5.938094,80.576134,no_mapped_loss,<NA>


# HRI preparation — visible cleaning and species join

This self-contained section consolidates the simple teaching notebooks into the existing workflow. No external processing script is called. Record-quality flags are preserved; no HRI weights or labels are invented. Run the cleaning save cell before the species join.

# 01 — Bird data understanding and cleaning for habitat-risk assessment

**Research aim:** assess habitat risk for Sri Lankan endemic birds using environmental indicators, an HRI, explanations and conservation recommendations.

This is the first teaching notebook in the corrected HRI workflow. All cleaning code is visible here; no external processing script is called. Run one cell at a time and inspect its output. The cells have intentionally not been executed for you.

Today we inspect the original observation table, standardise names and dates, interpret bird counts, and flag questionable records. We do **not** build HRI, train a model, or claim the dataset is final. Environmental columns in this original file need separate provenance/time checks; its elevation came from weather locations, not bird-coordinate terrain extraction. Earlier forest-loss modelling notebooks are a separate exploratory branch.

**Lecturer explanation:** “I preserve the raw data, show what each step changes, and separate uncertain records from valid values instead of silently replacing them.”

## 1. Import pandas and locate the project
`pandas` reads and cleans tables. `Path` builds file paths. Run from the project or this notebook folder.

In [22]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data/raw/observations/final_dataset_complete.csv").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the component4 project.")
    ROOT = ROOT.parent

print("Project folder:", ROOT)

Project folder: /Users/keerthikeerthi/Desktop/component4


## 2. Read the original file
Keep `raw` unchanged. Work on a copy called `clean`.

In [23]:
raw = pd.read_csv(ROOT / "data/raw/observations/final_dataset_complete.csv")
clean = raw.copy()
clean["source_row_id"] = raw.index + 1

print("Rows:", len(raw))
print("Columns:", len(raw.columns))
raw.head()

Rows: 102929
Columns: 23


,common_name,scientific_name,latitude,longitude,observation_date,year,month,observation_count,duration_minutes,effort_distance_km,...,temperature_min,temperature_mean,precipitation_sum,rain_sum,windspeed_max,windgusts_max,elevation,treecover2000,forest_loss,forest_loss_year
0,Red-backed Flameback,Dinopium psarodes,6.9035,80.1588,2010-01-07,2010,1,1,60.0,0.805,...,21.7,25.7,0.0,0.0,15.1,30.2,20.0,100,0,0
1,Sri Lanka Gray Hornbill,Ocyceros gingalensis,6.9035,80.1588,2010-01-07,2010,1,1,60.0,0.805,...,21.7,25.7,0.0,0.0,15.1,30.2,20.0,100,0,0
2,Sri Lanka Hanging-Parrot,Loriculus beryllinus,6.9035,80.1588,2010-01-07,2010,1,1,60.0,0.805,...,21.7,25.7,0.0,0.0,15.1,30.2,20.0,100,0,0
3,Chestnut-backed Owlet,Glaucidium castanotum,6.9035,80.1588,2010-01-07,2010,1,1,60.0,0.805,...,21.7,25.7,0.0,0.0,15.1,30.2,20.0,100,0,0
4,Crimson-fronted Barbet,Psilopogon rubricapillus,6.9035,80.1588,2010-01-07,2010,1,1,60.0,0.805,...,21.7,25.7,0.0,0.0,15.1,30.2,20.0,100,0,0


## 3. Inspect types and missing values
Missing effort does not mean zero effort. We will not fill missing values automatically.

In [24]:
raw.info()
raw.isna().sum().sort_values(ascending=False)

<class 'pandas.DataFrame'>
RangeIndex: 102929 entries, 0 to 102928
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   common_name           102929 non-null  str    
 1   scientific_name       102929 non-null  str    
 2   latitude              102929 non-null  float64
 3   longitude             102929 non-null  float64
 4   observation_date      102929 non-null  str    
 5   year                  102929 non-null  int64  
 6   month                 102929 non-null  int64  
 7   observation_count     102929 non-null  str    
 8   duration_minutes      91528 non-null   float64
 9   effort_distance_km    65872 non-null   float64
 10  protocol              102929 non-null  str    
 11  all_species_reported  102929 non-null  int64  
 12  temperature_max       102929 non-null  float64
 13  temperature_min       102929 non-null  float64
 14  temperature_mean      102929 non-null  float64
 15  precipitati

effort_distance_km      37057
duration_minutes        11401
common_name                 0
temperature_min             0
forest_loss                 0
treecover2000               0
elevation                   0
windgusts_max               0
windspeed_max               0
rain_sum                    0
precipitation_sum           0
temperature_mean            0
all_species_reported        0
temperature_max             0
scientific_name             0
protocol                    0
observation_count           0
month                       0
year                        0
observation_date            0
longitude                   0
latitude                    0
forest_loss_year            0
dtype: int64

## 4. Find repeated rows
An exact duplicate has the same value in every original column. The table lacks checklist identifiers, so identical rows are not proof of repeated ingestion. Flag them and preserve them in the main draft. A separate deduplicated candidate can be reviewed later; raw record counts are not independent survey counts.

In [25]:
clean["possible_duplicate"] = raw.duplicated(keep="first")
print("Extra identical rows:", clean["possible_duplicate"].sum())
raw.loc[raw.duplicated(keep=False)].head()

Extra identical rows: 18411


,common_name,scientific_name,latitude,longitude,observation_date,year,month,observation_count,duration_minutes,effort_distance_km,...,temperature_min,temperature_mean,precipitation_sum,rain_sum,windspeed_max,windgusts_max,elevation,treecover2000,forest_loss,forest_loss_year
13,Black-capped Bulbul,Rubigula melanictera,6.3721,80.4812,2010-01-01,2010,1,1,60.0,0.805,...,22.9,26.3,0.4,0.4,7.2,22.3,5.0,98,0,0
16,Black-capped Bulbul,Rubigula melanictera,6.3721,80.4812,2010-01-01,2010,1,1,60.0,0.805,...,22.9,26.3,0.4,0.4,7.2,22.3,5.0,98,0,0
20,Sri Lanka Hanging-Parrot,Loriculus beryllinus,6.3721,80.4812,2010-01-01,2010,1,1,60.0,0.805,...,22.9,26.3,0.4,0.4,7.2,22.3,5.0,98,0,0
22,Sri Lanka Hanging-Parrot,Loriculus beryllinus,6.3721,80.4812,2010-01-01,2010,1,1,60.0,0.805,...,22.9,26.3,0.4,0.4,7.2,22.3,5.0,98,0,0
26,Sri Lanka Blue-Magpie,Urocissa ornata,6.3721,80.4812,2010-01-01,2010,1,1,60.0,0.805,...,22.9,26.3,0.4,0.4,7.2,22.3,5.0,98,0,0


## 5. Standardise text
Remove accidental spaces without changing scientific taxonomy. Accepted-name matching and endemic-only selection will be a separate, sourced step.

In [26]:
clean["scientific_name"] = clean["scientific_name"].str.strip()
clean["common_name"] = clean["common_name"].str.strip()
clean["protocol"] = clean["protocol"].str.strip()

print("Scientific names:", clean["scientific_name"].nunique())
clean[["scientific_name", "common_name"]].drop_duplicates().head(10)

Scientific names: 25


,scientific_name,common_name
0,Dinopium psarodes,Red-backed Flameback
1,Ocyceros gingalensis,Sri Lanka Gray Hornbill
2,Loriculus beryllinus,Sri Lanka Hanging-Parrot
3,Glaucidium castanotum,Chestnut-backed Owlet
4,Psilopogon rubricapillus,Crimson-fronted Barbet
5,Psilopogon flavifrons,Yellow-fronted Barbet
7,Treron pompadora,Sri Lanka Green-Pigeon
9,Gallus lafayettii,Sri Lanka Junglefowl
10,Chrysocolaptes stricklandi,Crimson-backed Flameback
12,Argya cinereifrons,Ashy-headed Laughingthrush


## 6. Convert and check dates
Invalid dates become missing dates (`NaT`) and are flagged. Derive new year/month fields from the parsed date; keep the source year/month for comparison.

In [27]:
clean["observation_date_parsed"] = pd.to_datetime(
    clean["observation_date"], format="%Y-%m-%d", errors="coerce"
)
clean["invalid_date"] = clean["observation_date_parsed"].isna()
clean["observation_year"] = clean["observation_date_parsed"].dt.year.astype("Int64")
clean["observation_month"] = clean["observation_date_parsed"].dt.month.astype("Int64")
clean["date_fields_disagree"] = (
    clean["observation_year"].ne(clean["year"])
    | clean["observation_month"].ne(clean["month"])
).fillna(False)

print("Invalid dates:", clean["invalid_date"].sum())
print("Year/month disagreements:", clean["date_fields_disagree"].sum())
print("Date range:", clean["observation_date_parsed"].min(),
      "to", clean["observation_date_parsed"].max())

Invalid dates: 0
Year/month disagreements: 0
Date range: 2010-01-01 00:00:00 to 2023-06-17 00:00:00


## 7. Interpret bird counts
`X` indicates presence without a numeric count. Preserve it as a flag and leave its numeric count missing. Do not convert it to zero or one. Other unparseable, negative or fractional counts are flagged for review.

In [28]:
count_text = clean["observation_count"].astype("string").str.strip()
clean["presence_only"] = count_text.str.upper().eq("X").fillna(False)
clean["observation_count_numeric"] = pd.to_numeric(count_text, errors="coerce")

clean["count_needs_review"] = (
    (clean["observation_count_numeric"].isna() & ~clean["presence_only"])
    | clean["observation_count_numeric"].lt(0)
    | (clean["observation_count_numeric"].notna()
       & clean["observation_count_numeric"].mod(1).ne(0))
).fillna(False)

print("Presence-only records:", clean["presence_only"].sum())
print("Other count issues:", clean["count_needs_review"].sum())
clean[["observation_count", "observation_count_numeric", "presence_only"]].head()

Presence-only records: 12743
Other count issues: 0


,observation_count,observation_count_numeric,presence_only
0,1,1,False
1,1,1,False
2,1,1,False
3,1,1,False
4,1,1,False


## 8. Check coordinates
First validate global coordinate ranges. These checks do not establish whether a point lies in Sri Lanka; boundary matching is the next notebook. A point outside the working boundary needs investigation, not automatic deletion.

In [29]:
clean["latitude"] = pd.to_numeric(clean["latitude"], errors="coerce")
clean["longitude"] = pd.to_numeric(clean["longitude"], errors="coerce")
clean["invalid_coordinates"] = (
    ~clean["latitude"].between(-90, 90)
    | ~clean["longitude"].between(-180, 180)
)
print("Invalid or missing coordinates:", clean["invalid_coordinates"].sum())

Invalid or missing coordinates: 0


## 9. Check observation effort
Missing effort is retained. Negative effort and duration over 24 hours are flagged, not automatically corrected. Differences between survey protocols may explain some missing effort.

In [30]:
clean["effort_needs_review"] = (
    clean["duration_minutes"].lt(0)
    | clean["duration_minutes"].gt(1440)
    | clean["effort_distance_km"].lt(0)
)
print("Effort records requiring review:", clean["effort_needs_review"].sum())
clean.groupby("protocol")[["duration_minutes", "effort_distance_km"]].count()

Effort records requiring review: 20


,duration_minutes,effort_distance_km
protocol,,
Area,75,0
Banding,33,0
Historical,3578,1089
Incidental,0,8
Nocturnal Flight Call Count,2,0
Stationary,23047,30
Traveling,64792,64744
eBird Pelagic Protocol,1,1


## 10. Show the cleaning summary
Flags can overlap, so their counts must not be added as though they were separate records. This table is your evidence for the lecturer.

In [31]:
flags = ["possible_duplicate", "invalid_date", "date_fields_disagree",
         "count_needs_review", "invalid_coordinates", "effort_needs_review"]
summary = clean[flags].sum().rename("flagged_rows").to_frame()
clean["needs_review"] = clean[flags].any(axis=1)

print("Original rows:", len(raw))
print("Draft rows (all preserved):", len(clean))
print("Rows with one or more review flags:", clean["needs_review"].sum())
summary

Original rows: 102929
Draft rows (all preserved): 102929
Rows with one or more review flags: 18431


,flagged_rows
possible_duplicate,18411
invalid_date,0
date_fields_disagree,0
count_needs_review,0
invalid_coordinates,0
effort_needs_review,20


## 11. Inspect flagged records before making decisions
Do not delete all flagged rows at once. Document a decision for each type of issue. This notebook prepares a reviewable cleaning draft, not a final endemic-species dataset.

In [32]:
review = clean.loc[clean["needs_review"]].copy()
review[["scientific_name", "observation_date", "observation_count"] + flags].head(20)

,scientific_name,observation_date,observation_count,possible_duplicate,invalid_date,date_fields_disagree,count_needs_review,invalid_coordinates,effort_needs_review
16,Rubigula melanictera,2010-01-01,1,True,False,False,False,False,False
22,Loriculus beryllinus,2010-01-01,1,True,False,False,False,False,False
28,Urocissa ornata,2010-01-01,1,True,False,False,False,False,False
37,Psittacula calthrapae,2010-01-01,1,True,False,False,False,False,False
48,Dicrurus lophorinus,2010-01-01,1,True,False,False,False,False,False
76,Rubigula melanictera,2010-01-04,X,True,False,False,False,False,False
79,Dinopium psarodes,2010-01-03,X,True,False,False,False,False,False
80,Dinopium psarodes,2010-01-04,X,True,False,False,False,False,False
83,Loriculus beryllinus,2010-01-03,X,True,False,False,False,False,False
86,Treron pompadora,2010-01-03,X,True,False,False,False,False,False


## 12. Save the draft after you have reviewed the outputs
This cell writes only to a new HRI interim folder. It does not change the raw file or the earlier model dataset. The deduplicated candidate is separate and remains provisional.

In [33]:
OUT = ROOT / "data/interim/hri"
OUT.mkdir(parents=True, exist_ok=True)

clean.to_csv(OUT / "bird_observations_cleaning_draft.csv", index=False)
review.to_csv(OUT / "bird_observations_for_review.csv", index=False)
summary.to_csv(OUT / "cleaning_flag_summary.csv")
clean.loc[~clean["possible_duplicate"]].to_csv(
    OUT / "bird_observations_deduplicated_candidate.csv", index=False
)
print("Saved drafts to:", OUT)

Saved drafts to: /Users/keerthikeerthi/Desktop/component4/data/interim/hri


## What comes next?

1. Review the flagged records and document decisions.
2. Join the sourced species-trait table and resolve endemic-only eligibility without guessing unknown taxonomy or status.
3. Map retained observations to the country grid, preserving dates and observation-effort limitations.
4. Review environmental sources and construct justified habitat-risk indicators, then define and validate HRI.

No HRI weights, risk classes, bird absence labels or model-training targets are invented in this cleaning notebook. Its draft outputs must not be described as the final research dataset.

# 02 — Endemic-species verification

**Purpose:** connect cleaned bird records to sourced species traits and identify the endemic-only candidate dataset for habitat-risk assessment.

Run each cell yourself. All processing code is shown here. This notebook does not download traits, invent missing status, calculate weights or score HRI. The existing trait evidence is partially verified and reflects recorded source snapshots, not automatically current or historical assessment status.

Notebook 01 flags remain attached. We use its all-row cleaning draft; we do not silently adopt the deduplicated candidate or remove every flagged record. “Endemic candidate” means scope eligibility, not final record quality or completed trait verification.

## 1. Load the cleaning draft and trait evidence

In [34]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data/metadata/species_traits.csv").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside component4.")
    ROOT = ROOT.parent

birds = pd.read_csv(ROOT / "data/interim/hri/bird_observations_cleaning_draft.csv")
traits = pd.read_csv(ROOT / "data/metadata/species_traits.csv")
print("Bird records:", len(birds))
print("Species evidence rows:", len(traits))
traits[["scientific_name", "endemic_status", "iucn_category", "verification_status"]]

Bird records: 102929
Species evidence rows: 25


,scientific_name,endemic_status,iucn_category,verification_status
0,Argya cinereifrons,yes,VU,partially_verified
1,Batrachostomus moniliger,no,LC,partially_verified
2,Centropus chlororhynchos,yes,VU,partially_verified
3,Chrysocolaptes stricklandi,yes,LC,partially_verified
4,Columba torringtoniae,yes,VU,partially_verified
5,Dicrurus lophorinus,yes,LC,partially_verified
6,Dinopium psarodes,yes,LC,partially_verified
7,Galloperdix bicalcarata,yes,LC,partially_verified
8,Gallus lafayettii,yes,LC,partially_verified
9,Geokichla spiloptera,yes,NT,partially_verified


## 2. Check the species join key
Use the recorded scientific name for matching. Accepted names are retained separately; replacing names before verifying the crosswalk can lose matches or merge different taxonomic concepts.

In [35]:
birds["scientific_name"] = birds["scientific_name"].astype("string").str.strip()
traits["scientific_name"] = traits["scientific_name"].astype("string").str.strip()

assert traits["scientific_name"].notna().all(), "Trait names are missing."
assert traits["scientific_name"].is_unique, "Resolve duplicate trait keys before joining."

unmatched_names = birds.loc[
    ~birds["scientific_name"].isin(traits["scientific_name"]), "scientific_name"
].drop_duplicates()
print("Unmatched names:", unmatched_names.tolist())

Unmatched names: []


## 3. Join observations to species evidence
Many observations may belong to one species. `validate="many_to_one"` prevents a duplicate trait key from multiplying observation rows. Keep common names from observations and retain trait provenance.

In [36]:
trait_columns = [
    "scientific_name", "accepted_scientific_name", "endemic_status", "endemic_source",
    "iucn_category", "iucn_source", "iucn_snapshot_year", "iucn_assessment_year",
    "forest_dependence", "forest_dependence_source", "taxonomy_match_status",
    "verification_status", "inclusion_decision", "notes"
]
joined = birds.merge(
    traits[trait_columns], on="scientific_name", how="left",
    validate="many_to_one", indicator="trait_match"
)
assert len(joined) == len(birds), "The join changed the number of observations."
print(joined["trait_match"].value_counts())

trait_match
both          102929
left_only          0
right_only         0
Name: count, dtype: int64


## 4. Make the endemic-only scope decision visible
The study is about Sri Lankan endemic birds. Sourced `yes` records become candidates, `no` records stay in a separate outside-scope table, and unknown status remains pending. These new labels do not overwrite the master trait table or its pending review decisions.

In [37]:
status = joined["endemic_status"].astype("string").str.strip().str.lower()
joined["scope_decision"] = "pending_endemism_evidence"
joined.loc[status.eq("yes").fillna(False), "scope_decision"] = "endemic_candidate"
joined.loc[status.eq("no").fillna(False), "scope_decision"] = "outside_endemic_scope"

scope_summary = joined.groupby("scope_decision").agg(
    observation_rows=("scientific_name", "size"),
    species_names=("scientific_name", "nunique")
)
scope_summary

,observation_rows,species_names
scope_decision,,
endemic_candidate,101581,24
outside_endemic_scope,1348,1


## 5. Keep missing or conflicting traits visible
A missing IUCN category is not Least Concern. A category is not a numeric vulnerability weight. The stored taxonomy mismatch requires review; do not borrow the category of a broader parent taxon. Recorded source dates also cannot automatically be used as historical predictors.

In [38]:
joined["iucn_missing"] = joined["iucn_category"].isna()
joined["forest_dependence_missing"] = joined["forest_dependence"].isna()
joined["taxonomy_needs_review"] = (
    joined["taxonomy_match_status"].eq("taxon_concept_mismatch")
    | joined["taxonomy_match_status"].isna()
)
joined["trait_review_needed"] = (
    joined["iucn_missing"] | joined["forest_dependence_missing"]
    | joined["taxonomy_needs_review"]
    | joined["verification_status"].ne("verified")
)

species_review = joined.loc[joined["trait_review_needed"], trait_columns].drop_duplicates()
species_review[["scientific_name", "endemic_status", "iucn_category",
                "forest_dependence", "taxonomy_match_status", "verification_status"]]

,scientific_name,endemic_status,iucn_category,forest_dependence,taxonomy_match_status,verification_status
0,Dinopium psarodes,yes,LC,NaN,exact_name,partially_verified
1,Ocyceros gingalensis,yes,LC,NaN,exact_name,partially_verified
2,Loriculus beryllinus,yes,LC,NaN,exact_name,partially_verified
3,Glaucidium castanotum,yes,NT,NaN,exact_name,partially_verified
4,Psilopogon rubricapillus,yes,LC,NaN,exact_name,partially_verified
5,Psilopogon flavifrons,yes,LC,NaN,exact_name,partially_verified
7,Treron pompadora,yes,LC,NaN,exact_name,partially_verified
9,Gallus lafayettii,yes,LC,NaN,exact_name,partially_verified
10,Chrysocolaptes stricklandi,yes,LC,NaN,exact_name,partially_verified
12,Argya cinereifrons,yes,VU,High,exact_name,partially_verified


## 6. Separate the drafts without losing records
All earlier record-quality flags remain in these tables. Endemic candidates with incomplete traits remain visible rather than being assigned fabricated risk scores. Some traits may be unnecessary for a future HRI definition; decide that explicitly later.

In [39]:
endemic = joined.loc[joined["scope_decision"].eq("endemic_candidate")].copy()
outside_scope = joined.loc[joined["scope_decision"].eq("outside_endemic_scope")].copy()
pending = joined.loc[joined["scope_decision"].eq("pending_endemism_evidence")].copy()

assert len(endemic) + len(outside_scope) + len(pending) == len(birds)
print("Endemic candidate records:", len(endemic))
print("Endemic candidate species:", endemic["scientific_name"].nunique())
print("Record-quality review flags retained:", endemic["needs_review"].value_counts().to_dict())
print("Outside-scope species:", outside_scope["scientific_name"].drop_duplicates().tolist())

Endemic candidate records: 101581
Endemic candidate species: 24
Record-quality review flags retained: {False: 83426, True: 18155}
Outside-scope species: ['Batrachostomus moniliger']


## 7. Save only after reviewing the outputs
These are interim HRI preparation files, not final HRI labels or training data. The raw data and master trait evidence remain unchanged.

In [40]:
OUT = ROOT / "data/interim/hri"
OUT.mkdir(parents=True, exist_ok=True)
endemic.to_csv(OUT / "endemic_bird_candidates.csv", index=False)
outside_scope.to_csv(OUT / "birds_outside_endemic_scope.csv", index=False)
pending.to_csv(OUT / "birds_pending_endemism_review.csv", index=False)
species_review.to_csv(OUT / "species_traits_for_review.csv", index=False)
scope_summary.to_csv(OUT / "endemic_scope_summary.csv")
print("Saved reviewable drafts to:", OUT)

Saved reviewable drafts to: /Users/keerthikeerthi/Desktop/component4/data/interim/hri


## Explain this to your lecturer

“I joined the observations to a sourced species table without duplicating records. I separated non-endemic species because they are outside the study scope, preserved unknown cases and record-quality flags, and did not convert incomplete traits into vulnerability scores.”

**Next:** resolve cleaning decisions and map candidate observations to grid cells while preserving observation dates and effort. We then define supported environmental risk indicators and a justified HRI. We do not use the archived forest-loss label as HRI.

Read `data/metadata/species_traits_README.md` for sources and limitations. No fresh verification of current conservation status is claimed by this notebook.